In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%cudf.pandas.profile
### cell 22 ###

def get_n_grams(n_grams, top_n=10):
    results = []
    # cache the two columns once to avoid repeated __getitem__ calls
    dt_series = train['discourse_type']
    text_series = train['discourse_text']

    for dt in tqdm(dt_series.unique()):
        # boolean mask on the cached series
        texts = text_series[dt_series == dt]

        # build and run the vectorizer on CPU (as before) over the GPU-backed Series
        vec = CountVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(n_grams, n_grams)
        )
        bag = vec.fit_transform(texts)

        # sum counts and pull out into Python lists
        sums = bag.sum(axis=0)
        vocab = vec.vocabulary_
        words = list(vocab.keys())
        counts = [int(sums[0, idx]) for idx in vocab.values()]

        # build a single DataFrame with the Discourse_type column in one go
        df = pd.DataFrame({
            'Discourse_type': dt,
            'words': words,
            'counts': counts
        })
        # use nlargest instead of sort_values + head
        top = df.nlargest(top_n, 'counts')
        results.append(top)

    # concatenation stays the same
    return pd.concat(results)

# run for bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()